# MKNN Alignment — Subject-Averaged Embeddings

Same analyses as `analyze_mknn_hf.ipynb` but EEG embeddings are **averaged across
subjects before computing kNN**. This collapses idiosyncratic noise and preserves
the stimulus-locked (shared) signal, analogous to an ERP approach.

EEG arrays are `(n_layers, W, S, D)`. The S axis is averaged → `(n_layers, W, D)`,
then `precompute_knn_layers` is used (same as for vision embeddings). A single
Aristotelian MKNN score is produced per (EEG model size, vision model size) pair
instead of one per subject.

Analyses:
1. **Intramodal** within each EEG model (consecutive size pairs). reve is skipped.
2. **Cross-modal** EEG vs VideoMAE (k=5, k=10).
3. **Cross-modal** EEG vs DINOv2 (k=5, k=10).
4. **Cross-modal** EEG vs V-JEPA 2 (k=5, k=10).

In [ ]:
from pathlib import Path
import os, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from huggingface_hub import hf_hub_download

# ── Auth ──────────────────────────────────────────────────────────────────────
REPO = 'nitrox639/platonic-embeddings'
for _tp in (Path('../tokens/hf_token.txt'), Path('tokens/hf_token.txt'),
            Path('hf_token.txt'), Path.home() / 'hf_token.txt'):
    if _tp.exists():
        os.environ['HF_TOKEN'] = _tp.read_text().strip()
        break

def fetch(p):
    return hf_hub_download(REPO, p, repo_type='dataset')

def load_npz(p):
    return np.load(fetch(p), allow_pickle=True)

# ── Model registry ────────────────────────────────────────────────────────────
EEG = {
    'femba':       {'sizes': ['tiny', 'base', 'large'],  'family': 'femba_luna',
                    'fname': lambda s: f'eeg/femba/femba_{s}_tusl_layerwise.npz'},
    'luna':        {'sizes': ['base', 'large', 'huge'],  'family': 'femba_luna',
                    'fname': lambda s: f'eeg/luna/luna_{s}_layerwise.npz'},
    'neurolm':     {'sizes': ['b', 'l', 'xl'],           'family': 'neurolm',
                    'fname': lambda s: f'eeg/neurolm/neurolm_{s}_layerwise.npz'},
    'reve':        {'sizes': ['base', 'large'],          'family': 'reve',
                    'fname': lambda s: f'eeg/reve/reve_{s}_layerwise.npz'},
    'steegformer': {'sizes': ['small', 'base', 'large'], 'family': 'steegformer',
                    'fname': lambda s: f'eeg/steegformer/steegformer_{s}_layerwise.npz'},
}
VIDEOMAE_SIZES = ['base', 'large']
DINOV2_SIZES   = ['small', 'base', 'large', 'giant']
VJEPA2_SIZES   = ['large', 'huge', 'giant']

def vmpath(vs, fam):  return f'vision/videomae/videomae_{vs}__{fam}.npz'
def dnpath(vs, fam):  return f'vision/dinov2/dinov2_{vs}__{fam}.npz'
def vj2path(vs, fam): return f'vision/vjepa2/vjepa2_{vs}__{fam}.npz'

# ── Display metadata ──────────────────────────────────────────────────────────
PARAMS = {
    'femba':       {'tiny': 8,   'base': 48,  'large': 78},
    'luna':        {'base': 7,   'large': 43, 'huge': 311},
    'neurolm':     {'b': 254,    'l': 500,    'xl': 1696},
    'steegformer': {'small': 22, 'base': 86,  'large': 307},
    'reve':        {'base': 69,  'large': 408},
}
DISPLAY = {'femba': 'FEMBA', 'luna': 'LUNA', 'neurolm': 'NeuroLM',
           'steegformer': 'STEEGFormer', 'reve': 'REVE'}
def fmt_size(model, sz): return f'{sz} ({PARAMS[model][sz]}M)'

vm_colors  = {'base': 'steelblue', 'large': 'seagreen'}
dn_colors  = {'small': 'steelblue', 'base': 'seagreen',
              'large': 'darkorange', 'giant': 'mediumpurple'}
vj2_colors = {'large': 'steelblue', 'huge': 'seagreen', 'giant': 'darkorange'}

# ── Hyperparameters ───────────────────────────────────────────────────────────
K_MKNN = 5
K_PERM = 200
ALPHA  = 0.05

# ── kNN / mKNN primitives ─────────────────────────────────────────────────────
def l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def knn_1d(z, k):
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute').fit(z)
    _, idx = nn.kneighbors(z)
    return idx[:, 1:].astype(np.int32)

def mknn_1d(ia, ib):
    return float((ia[:, :, None] == ib[:, None, :]).any(-1).mean())

def precompute_knn_layers(emb, k):
    """Layerwise (n_layers, W, D) -> (n_layers, W, k).
    A 2-D (W, D) array is promoted to a single layer for back-compat."""
    emb = np.asarray(emb)
    if emb.ndim == 2:
        emb = emb[None]
    n_layers, W, _ = emb.shape
    idx = np.zeros((n_layers, W, k), dtype=np.int32)
    for l in range(n_layers):
        idx[l] = knn_1d(l2(emb[l]), k)
    return idx

def load_eeg_avg(fname, k):
    """Load EEG (n_layers, W, S, D), average over subjects, return kNN (n_layers, W, k)."""
    emb = load_npz(fname)['embeddings'].astype(np.float32)  # (n_layers, W, S, D)
    emb_avg = emb.mean(axis=2)                               # (n_layers, W, D)
    print(f'    loaded {emb.shape}  -> avg {emb_avg.shape}')
    return precompute_knn_layers(emb_avg, k)

# ── Aristotelian MKNN ─────────────────────────────────────────────────────────
def _calibrate(T_obs, T_null, alpha):
    tau   = float(np.quantile(T_null, 1 - alpha))
    s_cal = max(T_obs - tau, 0.0) / (1 - tau) if tau < 1 else 0.0
    p_val = float((1 + (T_null >= T_obs).sum()) / (len(T_null) + 1))
    return tau, s_cal, p_val

def aristotelian_cross(nn_a_layers, nn_b_layers, K=K_PERM, alpha=ALPHA, seed=42):
    """Cross-modal on averaged embeddings. Searches all layer-pair combinations.
    nn_a_layers: (LA, W, k)   nn_b_layers: (LB, W, k)."""
    LA, W, _ = nn_a_layers.shape
    LB = nn_b_layers.shape[0]
    rng = np.random.default_rng(seed)
    S_obs = np.array([[mknn_1d(nn_a_layers[la], nn_b_layers[lb]) for lb in range(LB)]
                      for la in range(LA)])
    T_obs = float(S_obs.max())
    T_null = np.zeros(K)
    for ki in range(K):
        perm = rng.permutation(W); inv = np.argsort(perm)
        b_perm = inv[nn_b_layers[:, perm, :]]
        T_null[ki] = max(mknn_1d(nn_a_layers[la], b_perm[lb])
                         for la in range(LA) for lb in range(LB))
    tau, s_cal, p_val = _calibrate(T_obs, T_null, alpha)
    return T_obs, s_cal, tau, p_val

# ── Generic cross-modal driver ────────────────────────────────────────────────
def run_cross_modal_avg(eeg_model, vision_arch, vision_sizes, vfname, k_mknn):
    fam       = EEG[eeg_model]['family']
    eeg_sizes = EEG[eeg_model]['sizes']
    vis_nn = {}
    for vs in vision_sizes:
        vemb = load_npz(vfname(vs, fam))['embeddings'].astype(np.float32)
        print(f'  {vision_arch}-{vs} ({fam}): {vemb.shape}')
        vis_nn[vs] = precompute_knn_layers(vemb, k_mknn)   # (LV, W, k)
    results = {}
    for esz in eeg_sizes:
        print(f'  {eeg_model}-{esz}: loading + averaging subjects...')
        nn_eeg = load_eeg_avg(EEG[eeg_model]['fname'](esz), k_mknn)  # (n_layers, W, k)
        LE = nn_eeg.shape[0]
        for vs in vision_sizes:
            LV = vis_nn[vs].shape[0]
            print(f'    cross  {eeg_model}-{esz} ({LE}L) x {vision_arch}-{vs} ({LV}L)'
                  f' = {LE*LV} layer pairs')
            T, sc, tau, p = aristotelian_cross(
                nn_eeg, vis_nn[vs], K=K_PERM, alpha=ALPHA, seed=42)
            results[(esz, vs)] = dict(T_obs=T, s_cal=sc, tau=tau, p=p)
            print(f'      uncal={T:.4f}  tau={tau:.4f}  cal={sc:.4f}  p={p:.4f}')
    return results

print('Setup complete. EEG models:', list(EEG.keys()))

## 1. Intramodal alignment per EEG model

Consecutive size pairs per model (reve skipped). Subject-averaged embeddings are
used; `aristotelian_cross` is applied between the two averaged EEG kNN tensors.

In [ ]:
INTRA_MODELS = [m for m in EEG if m != 'reve']

intra_results = {}   # intra_results[model] = {(a,b): {'T_obs','s_cal','tau','p'}}

for model in INTRA_MODELS:
    sizes = EEG[model]['sizes']
    pairs = list(zip(sizes[:-1], sizes[1:]))
    print(f'\n=== {model}  sizes={sizes}  pairs={pairs} ===')
    nn_by_size = {}
    for sz in sizes:
        print(f'  {model}-{sz}: loading + averaging subjects...')
        nn_by_size[sz] = load_eeg_avg(EEG[model]['fname'](sz), K_MKNN)
    intra_results[model] = {}
    for (a, b) in pairs:
        LA, LB = nn_by_size[a].shape[0], nn_by_size[b].shape[0]
        print(f'  pair {a}<->{b} ({LA}x{LB} layer pairs)')
        T, sc, tau, p = aristotelian_cross(
            nn_by_size[a], nn_by_size[b], K=K_PERM, alpha=ALPHA, seed=42)
        intra_results[model][(a, b)] = dict(T_obs=T, s_cal=sc, tau=tau, p=p)
        print(f'    uncal={T:.4f}  tau={tau:.4f}  cal={sc:.4f}  p={p:.4f}')

In [ ]:
ncols = 2
nrows = (len(INTRA_MODELS) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, INTRA_MODELS):
    pairs = list(intra_results[model].keys())
    pair_labels = [f'{fmt_size(model, a)}\n<->\n{fmt_size(model, b)}' for (a, b) in pairs]
    x = np.arange(len(pairs))
    cal_vals   = [intra_results[model][p]['s_cal'] for p in pairs]
    uncal_vals = [intra_results[model][p]['T_obs'] for p in pairs]
    width = 0.35
    ax.bar(x - width/2, uncal_vals, width, color='darkorange', alpha=0.8, label='uncalibrated (T_obs)')
    ax.bar(x + width/2, cal_vals,   width, color='steelblue',  alpha=0.8, label='calibrated (s_cal)')
    ax.set_xticks(x); ax.set_xticklabels(pair_labels, fontsize=9)
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)

for i in range(len(INTRA_MODELS), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Intramodal MKNN — subject-averaged embeddings (k={K_MKNN}, K_perm={K_PERM})',
             fontsize=12)
plt.tight_layout()
plt.show()

## 2. Cross-modal: EEG vs VideoMAE (k=5)

In [2]:
cross_vm = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs VideoMAE  (k={K_MKNN}) ===')
    cross_vm[model] = run_cross_modal_avg(model, 'videomae', VIDEOMAE_SIZES, vmpath, K_MKNN)


=== cross-modal femba vs VideoMAE  (k=5) ===
  videomae-base (femba_luna): (12, 2160, 768)
  videomae-large (femba_luna): (24, 2160, 1024)
  femba-tiny: loading + averaging subjects...


GatedRepoError: 401 Client Error. (Request ID: Root=1-6a0b3fb6-5f92e6236aed6bcd67acd2d8;6f3e0e17-46e6-4cbc-a593-5876b97962e6)

Cannot access gated repo for url https://huggingface.co/datasets/nitrox639/platonic-embeddings/resolve/main/eeg/femba/femba_tiny_tusl_layerwise.npz.
Access to dataset nitrox639/platonic-embeddings is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    width = 0.35 / len(VIDEOMAE_SIZES)
    offsets = np.linspace(-0.17, 0.17, len(VIDEOMAE_SIZES))
    for vs, off in zip(VIDEOMAE_SIZES, offsets):
        color = vm_colors[vs]
        cal_vals   = [cross_vm[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_vm[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.5, marker='o', markersize=7,
                label=f'VideoMAE-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'VideoMAE-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs VideoMAE  (k={K_MKNN}, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

### Same, with k=10

In [ ]:
cross_vm_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs VideoMAE  (k=10) ===')
    cross_vm_k10[model] = run_cross_modal_avg(model, 'videomae', VIDEOMAE_SIZES, vmpath, 10)

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VIDEOMAE_SIZES:
        color = vm_colors[vs]
        cal_vals   = [cross_vm_k10[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_vm_k10[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.5, marker='o', markersize=7,
                label=f'VideoMAE-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'VideoMAE-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs VideoMAE  (k=10, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

## 3. Cross-modal: EEG vs DINOv2 (k=5)

In [ ]:
cross_dn = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs DINOv2  (k={K_MKNN}) ===')
    cross_dn[model] = run_cross_modal_avg(model, 'dinov2', DINOV2_SIZES, dnpath, K_MKNN)

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in DINOV2_SIZES:
        color = dn_colors[vs]
        cal_vals   = [cross_dn[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_dn[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.2, marker='o', markersize=6,
                label=f'DINOv2-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=1.8, ls='--', marker='D', markersize=6,
                label=f'DINOv2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs DINOv2  (k={K_MKNN}, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

### Same, with k=10

In [ ]:
cross_dn_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs DINOv2  (k=10) ===')
    cross_dn_k10[model] = run_cross_modal_avg(model, 'dinov2', DINOV2_SIZES, dnpath, 10)

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in DINOV2_SIZES:
        color = dn_colors[vs]
        cal_vals   = [cross_dn_k10[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_dn_k10[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.2, marker='o', markersize=6,
                label=f'DINOv2-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=1.8, ls='--', marker='D', markersize=6,
                label=f'DINOv2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=6, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs DINOv2  (k=10, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

## 4. Cross-modal: EEG vs V-JEPA 2 (k=5)

In [ ]:
cross_vj2 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs V-JEPA 2  (k={K_MKNN}) ===')
    cross_vj2[model] = run_cross_modal_avg(model, 'vjepa2', VJEPA2_SIZES, vj2path, K_MKNN)

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VJEPA2_SIZES:
        color = vj2_colors[vs]
        cal_vals   = [cross_vj2[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_vj2[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.5, marker='o', markersize=7,
                label=f'V-JEPA2-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'V-JEPA2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs V-JEPA 2  (k={K_MKNN}, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()

### Same, with k=10

In [ ]:
cross_vj2_k10 = {}
for model in EEG:
    print(f'\n=== cross-modal {model} vs V-JEPA 2  (k=10) ===')
    cross_vj2_k10[model] = run_cross_modal_avg(model, 'vjepa2', VJEPA2_SIZES, vj2path, 10)

In [ ]:
models = list(EEG.keys())
ncols = 3
nrows = (len(models) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.2, nrows * 4.4))
axes_flat = axes.flatten()

for ax, model in zip(axes_flat, models):
    eeg_sizes = EEG[model]['sizes']
    x = np.arange(len(eeg_sizes))
    for vs in VJEPA2_SIZES:
        color = vj2_colors[vs]
        cal_vals   = [cross_vj2_k10[model][(esz, vs)]['s_cal'] for esz in eeg_sizes]
        uncal_vals = [cross_vj2_k10[model][(esz, vs)]['T_obs'] for esz in eeg_sizes]
        ax.plot(x, cal_vals,   color=color, lw=2.5, marker='o', markersize=7,
                label=f'V-JEPA2-{vs}  cal')
        ax.plot(x, uncal_vals, color=color, lw=2.0, ls='--', marker='D', markersize=7,
                label=f'V-JEPA2-{vs}  uncal')
    xtick_labels = [f'{sz} ({PARAMS[model][sz]}M)' for sz in eeg_sizes]
    ax.set_xticks(x); ax.set_xticklabels(xtick_labels, fontsize=9)
    ax.set_xlabel(f'{DISPLAY[model]} size')
    ax.set_ylabel('mKNN score', fontsize=10)
    ax.set_title(DISPLAY[model], fontsize=13, fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=7, ncol=2)

for i in range(len(models), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle(f'Cross-modal MKNN — EEG vs V-JEPA 2  (k=10, K_perm={K_PERM})  '
             'subject-averaged — solid = calibrated, dashed = uncalibrated',
             fontsize=12)
plt.tight_layout()
plt.show()